# Diffusion Models on MNIST — DDPM & DDIM

Three parts:
1. **Forward noising** — watch MNIST digits gradually turn into Gaussian noise
2. **Training** — train a DDPM to predict noise (MSE on $\epsilon$)
3. **Reverse denoising** — DDPM (stochastic) vs DDIM (deterministic, $10\times$ faster)


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

p = str(Path.cwd().parent) if Path.cwd().name == "apps" else str(Path.cwd())
if p not in sys.path:
    sys.path.insert(0, p)

from core.gen import DDIM, DDPM, NoiseScheduler, TimeConditionedUNet

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 128
LR = 2e-4
T = 200
EPOCHS = 10
BASE_CHANNELS = 32

torch.manual_seed(42)
print(f"Device: {DEVICE}, T={T}")

In [ ]:
# MNIST data loader
DATA_ROOT = (
    Path.cwd().parent / "assets" if Path.cwd().name == "apps" else Path.cwd() / "assets"
)
transform = transforms.Compose([transforms.ToTensor()])
train_set = datasets.MNIST(DATA_ROOT, train=True, download=False, transform=transform)
loader = DataLoader(train_set, BATCH_SIZE, shuffle=True, num_workers=2)
print(f"{len(train_set)} training images")

## 1. Forward Noising

Closed-form one-step noising:
$$x_t = \sqrt{\bar{\alpha}_t} \cdot x_0 + \sqrt{1 - \bar{\alpha}_t} \cdot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

$\bar{\alpha}_t$ decays from 1 → 0 as $t$ increases, so $x_t$ smoothly transitions from a clean image to pure Gaussian noise.

In [ ]:
scheduler = NoiseScheduler(timesteps=T)
scheduler.to(DEVICE)

# Visualise forward process: pick one image, show x_t at various t
x_0, _ = next(iter(loader))
x_0 = x_0[:1].to(DEVICE)  # single image

ts = [0, T // 4, T // 2, 3 * T // 4, T - 1]
fig, axes = plt.subplots(1, len(ts) + 1, figsize=(10, 2))

axes[0].imshow(x_0[0, 0].cpu(), cmap="gray")
axes[0].set_title("x₀ (clean)")
axes[0].axis("off")

for i, t_val in enumerate(ts):
    t = torch.tensor([t_val], device=DEVICE)
    x_t, _ = scheduler.add_noise(x_0, t)
    axes[i + 1].imshow(x_t[0, 0].cpu(), cmap="gray")
    axes[i + 1].set_title(f"t={t_val}", fontsize=9)
    axes[i + 1].axis("off")

fig.suptitle(f"Forward diffusion (T={T})", fontsize=12)
plt.tight_layout()

**Observation**: at $t=0$, $x_t$ is the original clean image; at $t=T$, it is nearly pure noise.

The goal of diffusion models: train a network to **reverse** this noising process.

## 2. Training

Training objective — simple MSE:

$$\mathcal{L} = \mathbb{E}_{t, x_0, \epsilon} \left[ \| \epsilon_\theta(x_t, t) - \epsilon \|^2 \right]$$

where $\epsilon_\theta$ is a **TimeConditionedUNet** — U-Net with FiLM time modulation + GroupNorm + SiLU.

Each step: sample random $t$ → noise $x_t$ → predict noise $\hat{\epsilon}$ → MSE loss.

In [ ]:
unet = TimeConditionedUNet(
    in_channels=1, out_channels=1, base_channels=BASE_CHANNELS, depth=3, time_dim=256
).to(DEVICE)
ddpm = DDPM(scheduler, unet).to(DEVICE)
opt = optim.AdamW(unet.parameters(), lr=LR)

print(f"UNet params: {sum(p.numel() for p in unet.parameters()) / 1e3:.1f}K")

In [ ]:
losses = []
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for x, _ in loader:
        x = x.to(DEVICE)
        opt.zero_grad()
        loss = ddpm.compute_loss(x)
        loss.backward()
        opt.step()
        epoch_loss += loss.item() * x.size(0)
    avg = epoch_loss / len(train_set)
    losses.append(avg)
    print(f"Epoch {epoch + 1:2d}/{EPOCHS}  loss={avg:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 2.5))
ax.plot(losses)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE")
ax.set_title("Training loss")
ax.grid(alpha=0.3)
plt.tight_layout()

## 3. Reverse Denoising — DDPM

Reverse process (DDPM Algorithm 2):

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\epsilon_\theta\right) + \sigma_t \cdot z, \quad z \sim \mathcal{N}(0, I)$$

No noise is added at the final step $t=0$ ($z=0$).

Below: start from pure noise $x_T$ and reverse-diffuse step by step (showing 10 evenly spaced intermediates).

In [ ]:
@torch.no_grad()
def show_reverse_steps(model, n_steps_shown=10):
    """Sample and show intermediate steps."""
    model.eval()
    scheduler = model.scheduler
    batch_size = 4
    device = next(model.unet.parameters()).device
    img_size = 28

    x_t = torch.randn(batch_size, 1, img_size, img_size, device=device)
    steps_to_show = set(
        np.linspace(0, scheduler.timesteps - 1, n_steps_shown, dtype=int)[::-1]
    )

    all_imgs = []
    for t in range(scheduler.timesteps - 1, -1, -1):
        t_batch = torch.full((batch_size,), t, device=device)
        noise_pred = model.unet(x_t, t_batch)
        x_t = model._reverse_step(x_t, t_batch, noise_pred)
        if t in steps_to_show:
            all_imgs.append(x_t.cpu())

    fig, axes = plt.subplots(
        batch_size, len(all_imgs), figsize=(len(all_imgs) * 1.2, batch_size * 1.2)
    )
    for row in range(batch_size):
        for col, img in enumerate(all_imgs):
            label = f"t={list(steps_to_show)[col]}" if row == 0 else ""
            axes[row, col].imshow(img[row, 0], cmap="gray")
            axes[row, col].axis("off")

    fig.suptitle("DDPM reverse: noise → digits (t decreasing left→right)", fontsize=12)
    plt.tight_layout()
    return x_t  # final samples


final_samples = show_reverse_steps(ddpm)

## 4. DDPM vs DDIM

| | DDPM | DDIM |
|---|---|---|
| Reverse step | stochastic ($+\sigma_t z$) | **deterministic** ($\sigma_t=0$) |
| Steps needed | $T$ (typically 1000) | $S \ll T$ (50–100) |
| Speed | baseline | **10–20× faster** |
| Same noise → same output | ❌ | ✅ |

DDIM uses the **predicted-$x_0$ formulation**:

$$x_t = \sqrt{\bar{\alpha}_{t-1}} \cdot \hat{x}_0 + \sqrt{1 - \bar{\alpha}_{t-1}} \cdot \epsilon_\theta, \quad \hat{x}_0 = \frac{x_t - \sqrt{1 - \bar{\alpha}_t}\epsilon_\theta}{\sqrt{\bar{\alpha}_t}}$$

Comparing DDPM (200 steps) vs DDIM at 4× speed (50 steps) and 10× speed (20 steps).

In [ ]:
ddim = DDIM(scheduler, unet).to(DEVICE)

torch.manual_seed(42)
np.random.seed(42)  # same noise seed for fair comparison
fixed_noise = torch.randn(4, 1, 28, 28, device=DEVICE)

# DDPM (T=200 steps)
with torch.no_grad():
    x_t = fixed_noise.clone()
    for t in range(scheduler.timesteps - 1, -1, -1):
        t_batch = torch.full((4,), t, device=DEVICE)
        x_t = ddpm._reverse_step(x_t, t_batch, unet(x_t, t_batch))
    ddpm_samples = x_t.cpu()

# DDIM (50 steps, 4x faster)
ddim_50 = ddim.sample(4, img_size=28, channels=1, num_steps=50, class_labels=None)

# DDIM (20 steps, 10x faster)
ddim_20 = ddim.sample(4, img_size=28, channels=1, num_steps=20, class_labels=None)

fig, axes = plt.subplots(4, 3, figsize=(5, 6))
titles = ["DDPM (200 steps)", "DDIM (50 steps, 4×)", "DDIM (20 steps, 10×)"]
for col in range(4):
    axes[col, 0].imshow(ddpm_samples[col, 0], cmap="gray")
    axes[col, 0].axis("off")
    axes[col, 1].imshow(ddim_50[col, 0].cpu(), cmap="gray")
    axes[col, 1].axis("off")
    axes[col, 2].imshow(ddim_20[col, 0].cpu(), cmap="gray")
    axes[col, 2].axis("off")
for col, t in enumerate(titles):
    axes[0, col].set_title(t, fontsize=9)
fig.suptitle("DDPM vs DDIM — same noise, different speed", fontsize=11)
plt.tight_layout()

## Summary

**Forward process:**
$$x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon$$

**Training loss (MSE on noise):**
$$\mathcal{L} = \mathbb{E}_{t,x_0,\epsilon}[\|\epsilon_\theta(x_t, t) - \epsilon\|^2]$$

**DDPM reverse:** stochastic, $T$ steps  
**DDIM reverse:** deterministic, $S \ll T$ steps, 10×+ speedup

**Building blocks:**
- `NoiseScheduler` — forward noising schedule
- `TimeConditionedUNet` — FiLM-modulated U-Net
- `DDPM` — training + stochastic sampling
- `DDIM` — deterministic accelerated sampling